In [3]:
import os
from tqdm import tqdm
from collections import defaultdict

import numpy as np
import pandas as pd
import random
from collections import deque

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import warnings

pytorch remind

In [20]:
data = [[1, 2, 3, 4],[5,6,7,8],[9,10,11,12],[13,14,15,16]]
x_data = torch.tensor(data)

In [21]:
x_data

tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12],
        [13, 14, 15, 16]])

In [28]:
x2 = torch.rand(3,4,5)

In [29]:
x2

tensor([[[0.4070, 0.9243, 0.4675, 0.8858, 0.8356],
         [0.8496, 0.9717, 0.8256, 0.5303, 0.0265],
         [0.8951, 0.2481, 0.8490, 0.9832, 0.2401],
         [0.9019, 0.7221, 0.0799, 0.2605, 0.5552]],

        [[0.4435, 0.3389, 0.6713, 0.5268, 0.3804],
         [0.7414, 0.1182, 0.3456, 0.1204, 0.8689],
         [0.4819, 0.2316, 0.6820, 0.6348, 0.9332],
         [0.0124, 0.8427, 0.2857, 0.2652, 0.1601]],

        [[0.9973, 0.6977, 0.9635, 0.3208, 0.3846],
         [0.5373, 0.6072, 0.6558, 0.4096, 0.3422],
         [0.7872, 0.3761, 0.1682, 0.4427, 0.8708],
         [0.9733, 0.7770, 0.0439, 0.5308, 0.0526]]])

In [31]:
tempx2 =  x2.sum(dim=2)

In [33]:
tempx2

tensor([[3.5203, 3.2037, 3.2154, 2.5196],
        [2.3609, 2.1945, 2.9635, 1.5661],
        [3.3638, 2.5520, 2.6451, 2.3776]])

In [32]:
tempx2.unsqueeze(2)

tensor([[[3.5203],
         [3.2037],
         [3.2154],
         [2.5196]],

        [[2.3609],
         [2.1945],
         [2.9635],
         [1.5661]],

        [[3.3638],
         [2.5520],
         [2.6451],
         [2.3776]]])

In [4]:
#nn.linear

m = nn.Linear(20,30)
x_data = torch.randn(128, 20)
output = m(x_data)



In [6]:
print('nn.Linear')
print(m)
print('----before linear')
print(x_data.shape)
print(x_data)
print('----after linear')
print(output.shape)
print(output)


nn.Linear
Linear(in_features=20, out_features=30, bias=True)
----before linear
torch.Size([128, 20])
tensor([[ 1.1547, -0.0410, -1.8951,  ...,  1.3443, -0.0331,  0.2883],
        [ 1.4968,  0.9884,  0.1093,  ...,  1.3539,  1.4189, -1.8829],
        [-0.1395,  1.4434,  1.6924,  ...,  0.6616, -0.2023, -1.4947],
        ...,
        [-0.9681,  2.4817, -0.8430,  ..., -0.1606,  1.7089, -0.3789],
        [-3.2677,  0.1684,  1.3422,  ...,  0.5966,  0.1965, -1.0859],
        [ 0.7163,  0.3147,  0.3641,  ..., -0.2162,  0.5523, -0.3782]])
----after linear
torch.Size([128, 30])
tensor([[-0.1925,  1.2170, -0.1466,  ..., -0.2130, -0.7892, -0.2140],
        [ 0.5353,  1.4273,  0.0093,  ..., -0.0567, -1.1015, -0.5707],
        [ 0.3114,  0.0044, -0.1546,  ..., -0.2238,  0.0262, -1.0823],
        ...,
        [ 0.3436,  0.6811, -0.8761,  ..., -1.3121, -0.2007, -0.9463],
        [-0.0379, -0.8612,  0.0870,  ..., -0.7289,  1.1851, -1.1775],
        [-0.0312,  0.0169,  0.4144,  ..., -0.3159, -0.3469,  0.

In [9]:
print(output.shape)

torch.Size([128, 30])


In [24]:
print(x_data.sum(dim=1))
#print(x_data.sum(dim=2))

tensor([10, 26, 42, 58])


Notes
graphsage train aggregator functions: train a set of aggregator functions that learn to aggregate feature information from a node's local neighborhood


In [35]:
#modified version of mean aggregator that replace step4,5 in Algo 1
# aggregate + activation by relu but no concatenation

#Q: output dim?
class MeanPoolingAggregator(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MeanPoolingAggregator, self).__init__()
        self.W = nn.Linear(input_dim, otuput_dim)
        self.activation = nn.ReLU()
        
    def forward(self, adj, x):
        #bmm: matrix multiplication that only work if shape is matched
        #adj is a tensor that contain all adj node features
        
        #matmul with self and neighbors
        #and
        #meanpooling (average pooling), result dimention is 2
        output = torch.bmm(adj, x) / adj.sum(dim = 2).unsqueeze(2)
        #apply activation
        output = self.W(output)
        output = self.activation(output)
        return output
        
        
        
        
    

In [1]:
url = 'https://files.grouplens.org/datasets/movielens/ml-latest-small.zip'

In [10]:


class GraphSAGELayer(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(GraphSAGELayer, self).__init__()
        self.Aggregator = MeanPoolingAggregator(input_dim, hidden_dim)
        
        #the input size for this linear transformation is twice the hidden dimention because the aggregation step
        #typically concatenates the node's own embedding with the aggregated information from its neighbor
        self.W = nn.Linear(input_dim + hidden_dim, hidden_dim) #why? 
        self.activation = nn.ReLU()
    
    def forward(self, adj, x):
        output = self.Aggregator(adj, x)
        output = torch.concat([x, output], dim = -1)
        output = self.W(output)
        output = self.activation(output)
        output = output / torch.linalg.norm(outptu, dim = 2).unsqueeze(2)
        return output
    
    
class GraphSAGE(nn.Module):
    def __init__(self, input_dim, hidden_dim, K):
        super(GraphSAGE, self).__init__()
        layers = []
        for _ in range(K):
            input_dim = input_dim
            layers.append(GraphSAGELayer(input_dim, hidden_dim))
            input_dim = hidden_dim
            hidden_dim = hidden_dim
            
        self.layers = nn.ModuleList(layers)
        self.L1 = nn.Linear(hidden_dim, hidden_dim, bias = True)
        self.L2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        
    def forward(self, nodes, adj, feature):
        output = feature[nodes]
        for layer in self.layers:
            output = layer(adj, output) #layer(adg, outpt) == GraphSAGELayer(inp_dim, hidden_dim)
        output = self.L1(output) #output shape should be hidden_dim x hidden_dim
        output = nn.ReLU()
        output = self.L2(output)
        return output
    
    